In [1]:
"""
DETESTS-Dis EDA
===============

"""

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
import warnings
import datasets
warnings.filterwarnings("ignore")

# ── load ──────────────────────────────────────────────────────────────────────
try:
    from datasets import load_dataset
    ds = load_dataset("CLiC-UB/DETESTS-Dis", split="train")
    df = ds.to_pandas()
    print(f"Loaded {len(df):,} rows from HuggingFace")
except Exception as e:
    print(f"Could not load from HuggingFace: {e}")
    raise

# ── style ─────────────────────────────────────────────────────────────────────
PALETTE = {
    "detests":   "#4A7FC1",
    "stereohoax": "#D85A30",
    "yes":  "#D85A30",
    "no":   "#4A7FC1",
    "agree": "#3B6D11",
    "disagree": "#BA7517",
}
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.edgecolor":   "#cccccc",
    "axes.spines.top":  False,
    "axes.spines.right": False,
})

SOFT_BINS = [0.0474, 0.2689, 0.7311, 0.9526]
BIN_LABELS = ["0/3\n(0.05)", "1/3\n(0.27)", "2/3\n(0.73)", "3/3\n(0.95)"]


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Label distributions
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Figure 1 — Label distributions", fontsize=15, fontweight="bold", y=0.98)

sources = df["source"].unique()
src_colors = [PALETTE["detests"], PALETTE["stereohoax"]]

# 1a — stereotype hard label by source
ax = axes[0, 0]
ct = df.groupby(["source", "stereotype"]).size().unstack(fill_value=0)
ct.plot(kind="bar", ax=ax, color=[PALETTE["no"], PALETTE["yes"]], edgecolor="white", width=0.6)
ax.set_title("Stereotype label by source")
ax.set_xlabel("")
ax.set_ylabel("Sentence count")
ax.set_xticklabels(ct.index, rotation=0)
ax.legend(["No stereotype (0)", "Stereotype (1)"], frameon=False)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=9)

# 1b — implicit hard label by source
ax = axes[0, 1]
ct2 = df[df["stereotype"] == 1].groupby(["source", "implicit"]).size().unstack(fill_value=0)
ct2.plot(kind="bar", ax=ax, color=[PALETTE["no"], PALETTE["yes"]], edgecolor="white", width=0.6)
ax.set_title("Implicit vs explicit (stereotype=1 only)")
ax.set_xlabel("")
ax.set_ylabel("Sentence count")
ax.set_xticklabels(ct2.index, rotation=0)
ax.legend(["Explicit (0)", "Implicit (1)"], frameon=False)

# 1c — stereotype × implicit joint heatmap
ax = axes[0, 2]
joint = df.groupby(["stereotype", "implicit"]).size().unstack(fill_value=0)
sns.heatmap(joint, annot=True, fmt="d", cmap="Blues", ax=ax,
            linewidths=0.5, cbar_kws={"shrink": 0.7})
ax.set_title("Stereotype × implicit joint distribution")
ax.set_xlabel("Implicit")
ax.set_ylabel("Stereotype")

# 1d — soft label distribution (stereotype_soft)
ax = axes[1, 0]
soft_counts = df["stereotype_soft"].round(4).value_counts().reindex(
    [round(b, 4) for b in SOFT_BINS], fill_value=0
)
bars = ax.bar(BIN_LABELS, soft_counts.values,
              color=[PALETTE["no"], "#8aabda", "#e09070", PALETTE["yes"]],
              edgecolor="white", width=0.6)
ax.set_title("Stereotype soft-label distribution")
ax.set_xlabel("Annotator votes (soft value)")
ax.set_ylabel("Sentence count")
for b, v in zip(bars, soft_counts.values):
    ax.text(b.get_x() + b.get_width()/2, v + 30, f"{v:,}",
            ha="center", va="bottom", fontsize=9)

# 1e — soft label distribution (implicit_soft)
ax = axes[1, 1]
soft_counts2 = df[df["stereotype"] == 1]["implicit_soft"].round(4).value_counts().reindex(
    [round(b, 4) for b in SOFT_BINS], fill_value=0
)
bars2 = ax.bar(BIN_LABELS, soft_counts2.values,
               color=[PALETTE["no"], "#8aabda", "#e09070", PALETTE["yes"]],
               edgecolor="white", width=0.6)
ax.set_title("Implicit soft-label distribution\n(stereotype=1 only)")
ax.set_xlabel("Annotator votes (soft value)")
ax.set_ylabel("Sentence count")


plt.tight_layout()
plt.savefig("fig1_label_distributions.png", dpi=150, bbox_inches="tight")
print("Saved fig1_label_distributions.png")
plt.close()





# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Text & conversational properties
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Figure 3 — Text and conversational properties", fontsize=15, fontweight="bold")

df["token_count"] = df["text"].str.split().str.len()

# 3a — sentence length distribution by stereotype label
ax = axes[0]
for label, color in [(0, PALETTE["no"]), (1, PALETTE["yes"])]:
    subset = df[df["stereotype"] == label]["token_count"]
    ax.hist(subset, bins=40, alpha=0.6, color=color, label=f"Stereotype={label}", density=True)
ax.set_title("Sentence length by stereotype label")
ax.set_xlabel("Token count")
ax.set_ylabel("Density")
ax.legend(frameon=False)
ax.set_xlim(0, 120)

# 3b — sentence length by soft label bin
ax = axes[1]
def assign_bin(v):
    v = round(v, 4)
    if v <= 0.05:   return "0/3"
    elif v <= 0.27: return "1/3"
    elif v <= 0.74: return "2/3"
    else:           return "3/3"

df["soft_bin"] = df["stereotype_soft"].apply(assign_bin)
order = ["0/3","1/3","2/3","3/3"]
bin_colors = [PALETTE["no"], "#8aabda", "#e09070", PALETTE["yes"]]
data_by_bin = [df[df["soft_bin"] == b]["token_count"].values for b in order]
bp = ax.boxplot(data_by_bin, labels=order, patch_artist=True, medianprops={"color":"white","linewidth":2})
for patch, color in zip(bp["boxes"], bin_colors):
    patch.set_facecolor(color)
ax.set_title("Sentence length by soft label bin")
ax.set_xlabel("Annotator agreement (votes)")
ax.set_ylabel("Token count")
ax.set_ylim(0, 100)

# 3c — stereotype rate by position within comment (DETESTS only)
ax = axes[2]
det = df[df["source"] == "detests"].copy()
det["position"] = det.groupby("comment_id").cumcount() + 1
pos_rate = det[det["position"] <= 10].groupby("position")["stereotype"].mean() * 100
ax.plot(pos_rate.index, pos_rate.values, marker="o", color=PALETTE["detests"], linewidth=2)
ax.fill_between(pos_rate.index, pos_rate.values, alpha=0.15, color=PALETTE["detests"])
ax.set_title("Stereotype rate by sentence position\nin comment (DETESTS, positions 1–10)")
ax.set_xlabel("Sentence position in comment")
ax.set_ylabel("% with stereotype")
ax.set_xticks(pos_rate.index)

plt.tight_layout()
plt.savefig("fig3_text_properties.png", dpi=150, bbox_inches="tight")
print("Saved fig3_text_properties.png")
plt.close()

print("\nAll figures saved. Open fig1_*.png, fig2_*.png, fig3_*.png to view.")

Loaded 9,906 rows from HuggingFace
Saved fig1_label_distributions.png
Saved fig3_text_properties.png

All figures saved. Open fig1_*.png, fig2_*.png, fig3_*.png to view.
